# Deep Learning · Session 31 · Task 1
## Your test, your table, your sentence

Prof. Dr. Judith Cerit · SRH University Munich · 04.11.2026

**What to do**
1. Replace the two lists below by **your** results: one number per run, the same seeds (or splits, or folds) in the same order.
2. Write your metric's name (accuracy, F1 score, Dice, …) and whether higher is better.
3. Say what changes from run to run: the seed, the split or the fold.
4. Run all cells. Copy the table, the sentence and the figure into your report, section 5 (Experiments).

The lists are filled with the example of Session 30 (GCN against GAT, 10 splits of Cora), so you can see what comes out.

In [ ]:
# Our project: read the test RSE of every seed from the run files (written by src/train.py).
# On Colab (Drive mounted) they are in the Drive folder; locally in results/runs/.
import glob, json, os
try:                                       # on Colab: connect Google Drive first
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

DATA = "electricity"                      # Dinesh: "solar"
BASELINE, IMPROVED = "lstnet", "lstnet_ar168"   # Dinesh: "lstnet", "lstnet_revin"
FOLDER = "/content/drive/MyDrive/lstnet-checkpoints"
if not os.path.isdir(FOLDER):
    FOLDER = "../results/runs"

def rse_per_seed(model):
    """Test RSE of seeds 0, 1, ..., 9 of one model, in seed order."""
    runs = {}
    for f in glob.glob(f"{FOLDER}/{DATA}_h24_{model}_skip24_seed*.json"):
        r = json.load(open(f))
        runs[int(r["run"].split("_seed")[-1])] = r["test_rse"]
    return [runs[s] for s in sorted(runs)]

baseline = rse_per_seed(BASELINE)         # your baseline, one number per seed
improved = rse_per_seed(IMPROVED)         # your improved model, the SAME seeds in the SAME order
metric = "test RSE"
higher_is_better = False                  # RSE: lower is better
baseline_name, improved_name = "LSTNet-Skip", "LSTNet + per-customer AR (168 h)"
run_name = "seed"                         # what changes from run to run
print("baseline:", [round(v, 4) for v in baseline])
print("improved:", [round(v, 4) for v in improved])

## Step 1 · Check the input

Same number of runs for both, and the runs belong together in pairs (run 1 of both used seed 0, …).

In [ ]:
assert len(baseline) == len(improved), "Both lists need one number per seed: the same length."
n = len(baseline)
print(f"{n} pairs of runs. Fewer than 5: no test can say much; with Wilcoxon, 5 pairs can never go below 0.06.")

## Step 2 · How large? (the effect size)

The difference per run, its mean and its spread: the size of your improvement.

In [ ]:
import numpy as np
b, i = np.array(baseline, float), np.array(improved, float)
diff = (i - b) if higher_is_better else (b - i)          # positive = the improved model is better
print(f"{baseline_name}: {b.mean():.2f} ± {b.std(ddof=1):.2f}")
print(f"{improved_name}: {i.mean():.2f} ± {i.std(ddof=1):.2f}")
print(f"difference per run: {np.round(diff, 2).tolist()}")
print(f"mean difference: {diff.mean():+.2f} ± {diff.std(ddof=1):.2f}   ·   better in {int((diff > 0).sum())} of {n} runs")

## Step 3 · How sure? (the tests)

The paired t-test and Wilcoxon compare run by run (Session 30). Below 0.05: significant.

In [ ]:
from scipy.stats import ttest_rel, wilcoxon
p_t = ttest_rel(i, b).pvalue
p_w = wilcoxon(i, b).pvalue
for name, p in (("paired t-test", p_t), ("Wilcoxon", p_w)):
    print(f"{name:14s} p = {p:.4f}   " + ("below 0.05: significant" if p < 0.05 else "not below 0.05: not significant"))

## Step 4 · For your report

The table and one sentence: how large, how sure, how many runs. Copy both.

In [ ]:
print(f"| Model | {metric}, {n} runs |")
print("|---|---|")
print(f"| {baseline_name} | {b.mean():.4f} ± {b.std(ddof=1):.4f} |")
print(f"| {improved_name} | {i.mean():.4f} ± {i.std(ddof=1):.4f} |")
verdict = "significant" if p_t < 0.05 else "not significant"
print(f"\n{improved_name} is {abs(diff.mean()):.4f} ± {diff.std(ddof=1):.4f} RSE {'better' if diff.mean() > 0 else 'worse'} than {baseline_name} "
      f"({n} runs with the same seeds; paired t-test p = {p_t:.4f}, {verdict}; Wilcoxon p = {p_w:.4f}).")

## Step 5 · Draw your runs (as Session 30, slide 12)

Three panels with the same scale, the runs in the same order:
1. **The baseline:** how much do your runs differ, only because of the seed, split or fold? (the bracket)
2. **The improved model:** does it go up and down with the baseline?
3. **Both together:** the baseline dashed, and one bar per run for the gap (blue: improved is better, orange: worse).

If the bracket in panel 1 is much larger than the gaps in panel 3, comparing run by run (paired) was the right choice.
The figure is saved as `paired_runs.png`: put it in your repository and your report.

In [ ]:
import matplotlib.pyplot as plt
x = np.arange(1, n + 1)                                   # run 1, 2, …, n on the horizontal axis
fig, axs = plt.subplots(1, 3, figsize=(13, 3.6), sharey=True)

# Panel 1: the baseline, with a bracket from its best run to its worst run
axs[0].plot(x, b, "o-", color="grey")
axs[0].vlines(n + 0.6, b.min(), b.max(), color="black")
axs[0].text(n + 0.9, (b.min() + b.max()) / 2, f"{b.max() - b.min():.4f}", va="center")
axs[0].set_xlim(0.5, n + 2)
axs[0].set_title(baseline_name, loc="left")

# Panel 2: the improved model alone
axs[1].plot(x, i, "o-", color="tab:blue")
axs[1].set_title(improved_name, loc="left")

# Panel 3: both together; one bar per run from the baseline to the improved model
axs[2].plot(x, b, "--", color="grey", label=baseline_name)
axs[2].plot(x, i, "o-", color="tab:blue", label=improved_name)
axs[2].vlines(x, b, i, colors=["tab:blue" if d > 0 else "tab:orange" for d in diff], linewidth=3)
axs[2].set_title(f"{improved_name} vs {baseline_name}: gap {diff.mean():+.4f} on average", loc="left")
axs[2].legend(frameon=False)

for ax in axs:
    ax.set_xticks(x); ax.set_xlabel(run_name)
axs[0].set_ylabel(metric)
plt.tight_layout(); plt.savefig(f"paired_runs_{DATA}.png", dpi=150); plt.show()